In [ ]:
import math
import torch
import gpytorch
from matplotlib import pyplot as plt
import h5py
import numpy as np
import random
import bacco

%matplotlib inline
%load_ext autoreload
%autoreload 2

In [ ]:
import emcee
import corner

In [ ]:
import os
os.chdir("/cosmos_storage/home/fgmaion/MTNG-resims/src")
import utils

In [ ]:
plt.rcParams["font.family"] = "serif"
plt.rcParams["mathtext.fontset"] = "dejavuserif"

### Get Parameters

In [ ]:
wind_en_or      = []
wind_vel_or     = []
rho_rec_or      = []
sf_ts_or        = []
ef_kin_or       = []
ef_high_or      = []
f_re_or         = []

for i in range(31):
    if i<30:
        filename = "/cosmos_storage/simulations/TNG_Family/MN5_resims/param_LH/param_MTNG-hydro_{:d}.txt".format(i)
    else:
        filename = "/cosmos_storage/simulations/TNG_Family/MN5_resims/param_LH/param_MTNG-hydro.txt"

    with open(filename, 'r') as f:
        for line in f.readlines():
            if len(line.split())!=0:
                if line.split()[0] == 'WindEnergyIn1e51erg':
                    wind_en_or.append(float(line.split()[1]))
                if line.split()[0] == 'VariableWindVelFactor':
                    wind_vel_or.append(float(line.split()[1]))
                if line.split()[0] == 'WindFreeTravelDensFac':
                    rho_rec_or.append(float(line.split()[1]))
                if line.split()[0] == 'MaxSfrTimescale':
                    sf_ts_or.append(float(line.split()[1]))
                if line.split()[0] == 'RadioFeedbackFactor':
                    ef_kin_or.append(float(line.split()[1]))
                if line.split()[0] == 'BlackHoleFeedbackFactor':
                    ef_high_or.append(float(line.split()[1]))
                if line.split()[0] == 'RadioFeedbackReiorientationFactor':
                    f_re_or.append(float(line.split()[1]))

rho_rec_or = np.log10(rho_rec_or)
ef_kin_or = np.log10(ef_kin_or)

wind_en   = (np.asarray(wind_en_or) - np.mean(wind_en_or)) / np.std(wind_en_or)
wind_vel  = (np.asarray(wind_vel_or) - np.mean(wind_vel_or)) / np.std(wind_vel_or)
rho_rec   = (np.asarray(rho_rec_or) - np.mean(rho_rec_or)) / np.std(rho_rec_or)
sf_ts     = (np.asarray(sf_ts_or) - np.mean(sf_ts_or)) / np.std(sf_ts_or)
ef_kin    = (np.asarray(ef_kin_or) - np.mean(ef_kin_or)) / np.std(ef_kin_or)
ef_high   = (np.asarray(ef_high_or) - np.mean(ef_high_or)) / np.std(ef_high_or)
f_re      = (np.asarray(f_re_or) - np.mean(f_re_or)) / np.std(f_re_or)

In [ ]:
name_list = ['LH_{:d}'.format(i) for i in range(30)] + ['fiducial']

In [ ]:
fid_pars = np.array([wind_en[30], wind_vel[30], rho_rec[30], sf_ts[30], ef_kin[30], ef_high[30], f_re[30]])

weights = (0.1, 1, 0.1, 1, 0.1, 0.1, 0.1)

dist = np.zeros(31)
for i in range(len(name_list)):
    pars_i = np.array([wind_en[i], wind_vel[i], rho_rec[i], sf_ts[i], ef_kin[i], ef_high[i], f_re[i]])
    dist[i] = np.sum((weights*(pars_i - fid_pars))**2)

In [ ]:
def pars(i, mstar):

    arr = np.vstack( ( mstar, np.ones(len(mstar)) * wind_en[i],\
                        np.ones(len(mstar)) * wind_vel[i],\
                        np.ones(len(mstar)) * rho_rec[i],\
                        np.ones(len(mstar)) * sf_ts[i],\
                        np.ones(len(mstar)) * ef_kin[i],\
                        np.ones(len(mstar)) * ef_high[i],\
                        np.ones(len(mstar)) * f_re[i])).T

    return arr

In [ ]:
Nbins_smf = 10

zoom_smf = {}
for i in range(len(name_list)):
    zoom_smf[name_list[i]] = np.load("/cosmos_storage/home/fgmaion/MTNG-resims/results/smf/new_smf_{}_Nbins{:d}.npy".format(name_list[i], Nbins_smf), allow_pickle=True)[0]

In [ ]:
# Filter NaNs
for i in range(len(name_list)):
    mask = ~np.isnan(zoom_smf[name_list[i]]['smf'][0]) & ~np.isnan(zoom_smf[name_list[i]]['mstar'][0])
    
    zoom_smf[name_list[i]]['smf'][0] = zoom_smf[name_list[i]]['smf'][0][mask]
    zoom_smf[name_list[i]]['mstar'][0] = zoom_smf[name_list[i]]['mstar'][0][mask]

In [ ]:
train_sel = np.argsort(dist)[::-1][3:31]
test_sel = list(set(range(31)) - set(train_sel))

In [ ]:
#del smf_global, arr_global
pars_global = pars(train_sel[0], np.log10(zoom_smf[name_list[train_sel[0]]]['mstar'][0]) )
smf_global = np.log10(zoom_smf[name_list[train_sel[0]]]['smf'][0])

for i in range(len(train_sel)):
    mstar = np.log10(zoom_smf[name_list[train_sel[i]]]['mstar'][0])

    arr = pars(train_sel[i], mstar)

    pars_global = np.vstack((pars_global, arr))

    smf_global = np.hstack((smf_global, np.log10(zoom_smf[name_list[train_sel[i]]]['smf'][0])))

In [ ]:
train_x = torch.asarray(pars_global, dtype=torch.float)
train_y = torch.asarray(smf_global, dtype=torch.float)

In [ ]:
torch.set_num_threads(8)

In [ ]:
smf_draws = np.load("/cosmos_storage/home/fgmaion/MTNG-resims/results/smf/smf_draws/smf_draws100_nbins10.npy", allow_pickle=True).item()

err_smf = np.std(smf_draws['ens_smf'], axis=0)
err_log_smf = torch.asarray(err_smf / np.mean(smf_draws['ens_smf'], axis=0) / np.log(10), dtype=torch.float)

### Build the GP Model

In [ ]:
# We will use the simplest form of GP model, exact inference
class ExactGPModel(gpytorch.models.ExactGP):
    def __init__(self, train_x, train_y, likelihood):
        super(ExactGPModel, self).__init__(train_x, train_y, likelihood)
        self.mean_module = gpytorch.means.ConstantMean()
        self.covar_module = gpytorch.kernels.ScaleKernel(gpytorch.kernels.RBFKernel(ard_num_dims=8))

    def initialize(self):
        def inv_softplus(x):
            x = torch.as_tensor(x, dtype=torch.float)
            return torch.log(torch.expm1(x))
        
        with torch.no_grad():
            # Lengthscales: random in [0.5, 1.5]
            ls = 0.5 + torch.rand_like(self.covar_module.base_kernel.raw_lengthscale)
            self.covar_module.base_kernel.raw_lengthscale.copy_(inv_softplus(ls))
            
            # Outputscale: random in [0.5, 2.0]
            os_val = 0.5 + 1.5 * torch.rand(()).item()
            self.covar_module.raw_outputscale.fill_(inv_softplus(os_val).item())
            
            # Mean: small random
            self.mean_module.raw_constant.fill_((torch.randn(()) * 0.1).item())
            
            # Additional noise: random in [0.01, 0.1]
            # nn_val = 0.005
            # self.likelihood.second_noise_covar.raw_noise.fill_(inv_softplus(nn_val).item())
    
    def forward(self, x):
        mean_x = self.mean_module(x)
        covar_x = self.covar_module(x)
        return gpytorch.distributions.MultivariateNormal(mean_x, covar_x)

# initialize likelihood and model
# likelihood = gpytorch.likelihoods.GaussianLikelihood(noise_constraint=gpytorch.constraints.GreaterThan(1e-3))
likelihood = gpytorch.likelihoods.FixedNoiseGaussianLikelihood(noise=err_log_smf**2, learn_additional_noise=True)
model = ExactGPModel(train_x, train_y, likelihood)

#### Train it

In [ ]:
# this is for running the notebook in our testing framework
import os
import copy
smoke_test = ('CI' in os.environ)
#training_iter = 2 if smoke_test else 200

n_restarts = 10
steps_per_restart = 1000

best_state = None
best_loss = float('inf')

for r in range(n_restarts):
    print(f"\n=== Restart {r+1}/{n_restarts} ===")
    model.initialize()

    # Find optimal model hyperparameters
    model.train()
    likelihood.train()

    # Use the adam optimizer
    optimizer = torch.optim.Adam(model.parameters(), lr=0.01)  # Includes GaussianLikelihood parameters

    # "Loss" for GPs - the marginal log likelihood
    mll = gpytorch.mlls.ExactMarginalLogLikelihood(likelihood, model)

    for i in range(steps_per_restart):
        # Zero gradients from previous iteration
        optimizer.zero_grad()
        # Output from model
        output = model(train_x)
        # Calc loss and backprop gradients
        loss = -torch.sum(mll(output, train_y))
        loss.backward()
        optimizer.step()

    # compute final loss for this restart
    with torch.no_grad():
        final_output = model(train_x)
        final_loss = float(-mll(final_output, train_y))

    print(f"Final loss restart {r+1}: {final_loss:.4f}")

    # store best
    if final_loss < best_loss:
        best_loss = final_loss
        best_state = {
            'model': copy.deepcopy(model.state_dict()),
            'likelihood': copy.deepcopy(likelihood.state_dict()),
        }

In [ ]:
# ---- restore best ----
model.load_state_dict(best_state['model'])
likelihood.load_state_dict(best_state['likelihood'])

In [ ]:
model.likelihood.second_noise_covar.noise = 0.1**2

In [ ]:
print(model.covar_module.base_kernel.lengthscale)
print(model.covar_module.outputscale)
print(torch.sqrt(model.likelihood.noise_covar.noise))

extra_noise = model.likelihood.second_noise_covar.noise
print(f"Learned additional noise (variance): {extra_noise.item():.6f}")
print(f"Learned additional noise (std): {extra_noise.item()**0.5:.6f}")

In [ ]:
# Get into evaluation (predictive posterior) mode
model.eval()
likelihood.eval()

In [ ]:
# Stellar Mass Function from  GAMA (corrected to h=0.7)
# https://arxiv.org/pdf/2203.08539
# For SDSS correction, add 0.0807 dex to number density

GAMA = np.array([
    [6.875, -0.691, 0.176],
    [7.125, -1.084, 0.125],
    [7.375, -1.011, 0.071],
    [7.625, -1.349, 0.092],
    [7.875, -1.287, 0.079],
    [8.125, -1.544, 0.071],
    [8.375, -1.669, 0.045],
    [8.625, -1.688, 0.032],
    [8.875, -1.795, 0.024],
    [9.125, -1.886, 0.020],
    [9.375, -2.055, 0.014],
    [9.625, -2.142, 0.010],
    [9.875, -2.219, 0.009],
    [10.125, -2.274, 0.009],
    [10.375, -2.292, 0.009],
    [10.625, -2.361, 0.010],
    [10.875, -2.561, 0.013],
    [11.125, -2.922, 0.019],
    [11.375, -3.414, 0.032],
    [11.625, -4.704, 0.138]
])

GAMA_corr = np.zeros((GAMA.shape[0], 4))
GAMA_corr[:,0] = GAMA[:,0] - 2*np.log10(0.7)
GAMA_corr[:,1] = GAMA[:,1] + 3*np.log10(0.7)


def model_GAMA(GAMA_corr, b_star, b_cv):

    GAMA_corr = np.copy(GAMA_corr)
    GAMA_corr[:,1] = GAMA_corr[:,1] + np.log10(b_cv)
    GAMA_corr[:,0] = GAMA_corr[:,0] + b_star

    return GAMA_corr

In [ ]:
with torch.no_grad(), gpytorch.settings.fast_pred_var():
    # Initialize plot
    f, ax = plt.subplots(5, 5, figsize=(30, 30), dpi=150)

    for i in range(5):
        for j in range(5):
            test_x = torch.asarray(pars(train_sel[5*i+j], np.log10(zoom_smf[name_list[train_sel[5*i+j]]]['mstar'][0])), dtype=torch.float)
            observed_pred = likelihood(model(torch.asarray(test_x, dtype=torch.float)), noise=torch.asarray(err_log_smf[-test_x.shape[0]:]**2, dtype=torch.float))

            ax[i,j].set_title('Simulation {:s}'.format(name_list[train_sel[5*i+j]]), fontsize=16)
            ax[i,j].errorbar(GAMA_corr[:,0], GAMA_corr[:,1], yerr=GAMA[:,2], color='r', marker='o', capsize=3, ms=8, fillstyle='none', ls='', label='GAMA (Driver+ 2019)')

            ax[i,j].set_xlim(9.5,12.5)
            ax[i,j].set_ylim(-6, -1.8)

            if j==0:
                ax[i,j].set_ylabel('$\log_{10}(\Phi/[M_{\odot}\mathrm{Mpc}^{-3}])$', fontsize=16)
            if i==4:
                ax[i,j].set_xlabel('$M_*[M_{\odot}]$', fontsize=16)

            # Get upper and lower confidence bounds
            lower, upper = observed_pred.confidence_region()
            # Plot training data as black stars
            ax[i,j].plot(np.log10(zoom_smf[name_list[train_sel[5*i+j]]]['mstar'][0]), np.log10(zoom_smf[name_list[train_sel[5*i+j]]]['smf'][0]), 'b*')

            # Plot predictive means as blue line
        #    for i in range(10):
        #        ax.plot(mstar_test, smf_test.sample().numpy(), 'b', ls='--', lw=0.5)
            ax[i,j].plot(test_x[:,0], observed_pred.mean.numpy(), 'b')

            # Shade between the lower and upper confidence bounds
            ax[i,j].fill_between(test_x[:,0].numpy(), observed_pred.mean.numpy()-observed_pred.stddev.numpy(), observed_pred.mean.numpy()+observed_pred.stddev.numpy(), color='b', alpha=0.5, edgecolor=None)

ax[0,0].legend(loc='lower left', fontsize=18)

In [ ]:
gsmf = np.loadtxt('/cosmos_storage/home/fgmaion/MTNG-resims/data/GAMA_SDSS_stitched_GSMF_h0p6774.csv',
                  delimiter=',', comments='#', skiprows=33,
                  usecols=(0, 1, 2))

with torch.no_grad(), gpytorch.settings.fast_pred_var():
    # Initialize plot
    f, ax = plt.subplots(2, 3, figsize=(15, 10), dpi=100)
    plt.subplots_adjust(wspace=0.15, hspace=0.3)

    for i in range(2):
        for j in range(3):
            test_x = torch.asarray(pars(test_sel[3*i+j], np.log10(zoom_smf[name_list[test_sel[3*i+j]]]['mstar'][0])), dtype=torch.float)
            observed_pred = likelihood(model(torch.asarray(test_x, dtype=torch.float)), noise=torch.asarray(err_log_smf[-test_x.shape[0]:]**2, dtype=torch.float))
        
            ax[i,j].set_xlim(9,12.5)
            ax[i,j].set_ylim(-6, -1.8)

            ax[i,j].set_title('Simulation {:s}'.format(name_list[test_sel[3*i+j]]), fontsize=16)
            ax[i,j].errorbar(gsmf[:,0], gsmf[:,1], yerr=gsmf[:,2], color='k', marker='o', capsize=3, ms=6, fillstyle='none', ls='', label='GAMA+SDSS')

            if j==0:
                ax[i,j].set_ylabel('$\log_{10}(\Phi/[\mathrm{Mpc}^{-3}\mathrm{dex}^{-1}])$', fontsize=16)
            
            ax[i,j].set_xlabel('$\log_{10}(M_*/M_{\odot})$', fontsize=16)

            # Get upper and lower confidence bounds
            lower, upper = observed_pred.confidence_region()

            # Shade between the lower and upper confidence bounds
            ax[i,j].fill_between(test_x[:,0].numpy(), observed_pred.mean.numpy()-observed_pred.stddev.numpy(), observed_pred.mean.numpy()+observed_pred.stddev.numpy(), color='C0', alpha=0.4, edgecolor=None)
            ax[i,j].plot(test_x[:,0], observed_pred.mean.numpy(), color='#0077BB', lw=2, label="GP Prediction", alpha=0.9)

            # Plot test data as blue squares
            ax[i,j].plot(np.log10(zoom_smf[name_list[test_sel[3*i+j]]]['mstar'][0]), np.log10(zoom_smf[name_list[test_sel[3*i+j]]]['smf'][0]), 's', color='#0077BB', label="Test Data")

ax[0,0].legend(loc='lower left', fontsize=12)

plt.savefig("/cosmos_storage/home/fgmaion/MTNG-resims/results/testing_plots/smf_test.pdf", bbox_inches='tight')

In [ ]:
gsmf = np.loadtxt('/cosmos_storage/home/fgmaion/MTNG-resims/data/GAMA_SDSS_stitched_GSMF_h0p6774.csv',
                  delimiter=',', comments='#', skiprows=33,
                  usecols=(0, 1, 2))

with torch.no_grad(), gpytorch.settings.fast_pred_var():
    # Initialize plot
    f, ax = plt.subplots(figsize=(5.5, 5), dpi=100)
    plt.subplots_adjust(wspace=0.15, hspace=0.3)

    ax.set_ylabel('$\log_{10}(\Phi/[\mathrm{Mpc}^{-3}\mathrm{dex}^{-1}])$', fontsize=16)
    ax.set_xlabel('$\log_{10}(M_*/M_{\odot})$', fontsize=16)
    
    ax.fill_between(np.linspace(9,12.5,9), -2*(0.1+err_log_smf), 2*(0.1+err_log_smf), alpha=0.2, color='C0', edgecolor=None)
    ax.fill_between(np.linspace(9,12.5,9), -(0.1+err_log_smf), 0.1+err_log_smf, alpha=0.5, color='C0', edgecolor=None)

    for i in range(2):
        for j in range(3):
            test_x = torch.asarray(pars(test_sel[3*i+j], np.log10(zoom_smf[name_list[test_sel[3*i+j]]]['mstar'][0])), dtype=torch.float)
            observed_pred = likelihood(model(torch.asarray(test_x, dtype=torch.float)), noise=torch.asarray(err_log_smf[-test_x.shape[0]:]**2, dtype=torch.float))        

            # Get upper and lower confidence bounds
            lower, upper = observed_pred.confidence_region()

            ax.plot(np.log10(zoom_smf[name_list[test_sel[3*i+j]]]['mstar'][0]), np.log10(zoom_smf[name_list[test_sel[3*i+j]]]['smf'][0]) - observed_pred.mean.numpy(),  color='C'+str(i*3+j), lw=3)

ax.legend(loc='lower left', fontsize=12)

### Now look at model trained with full data

In [ ]:
os.chdir("/cosmos_storage/home/fgmaion/MTNG-resims/scripts/train")
from GP_models import SMF_Model, fgas_Model

In [ ]:
model_smf = torch.load("/cosmos_storage/home/fgmaion/MTNG-resims/gp_train_results/full_model_smf.pth")
likelihood_smf = torch.load("/cosmos_storage/home/fgmaion/MTNG-resims/gp_train_results/full_likelihood_smf.pth")

model_smf.eval()
likelihood_smf.eval()

In [ ]:
def get_theta_1p_vars(param_num, mstar, nvars):
    fid_theta = np.array([wind_en[30], wind_vel[30], rho_rec[30], sf_ts[30], ef_kin[30], ef_high[30], f_re[30]])

    theta_1p_vars = {}

    delta = 2 / nvars
    for i in range(nvars+1):
        par_array = np.copy(fid_theta)
        par_array[param_num] += (i - nvars//2) * delta
        theta_1p_vars[i] = np.vstack( ( mstar, np.ones(len(mstar)) * par_array[0],\
                                        np.ones(len(mstar)) * par_array[1],\
                                        np.ones(len(mstar)) * par_array[2],\
                                        np.ones(len(mstar)) * par_array[3],\
                                        np.ones(len(mstar)) * par_array[4],\
                                        np.ones(len(mstar)) * par_array[5],\
                                        np.ones(len(mstar)) * par_array[6])).T

    return theta_1p_vars

In [ ]:
fig, ax = plt.subplots(2, 4, dpi=200, sharey=False, figsize=(15,7))

# Paul Tol's discrete rainbow scheme - selecting 4 well-separated colors
# for the -2Δ, -Δ, +Δ, +2Δ variations (Fiducial stays black)
tol_rainbow = ['#3F60AE', '#6DB388', '#E68B33', '#D33C2A']  # blue, green, orange, red

mstar_test = np.linspace(9,12.5,50)
names = ['Wind Energy', 'Wind Velocity', 'Density for Recoupling', 'Max SFR Timescale', 'AGN Kinetic Feedback', 'AGN High Accretion Feedback', 'AGN Reorientation Factor']

for ax_i in np.ndarray.flatten(ax):
    ax_i.set_ylim(-0.4,0.4)
    # Thicker plot edges
    for spine in ax_i.spines.values():
        spine.set_linewidth(2.5)

    # Major and minor ticks
    ax_i.tick_params(axis='both', which='major', width=2.5, length=8, labelsize=12, direction='in', right=True, top=True)
    ax_i.tick_params(axis='both', which='minor', width=1.5, length=4, direction='in', right=True, top=True)
    ax_i.minorticks_on()

labels=['$-2\Delta$', '$-\Delta$', 'Fiducial', '$+\Delta$', '$+2\Delta$']

# Map n=0,1,3,4 to tol_rainbow indices 0,1,2,3
color_map = {0: tol_rainbow[0], 1: tol_rainbow[1], 3: tol_rainbow[2], 4: tol_rainbow[3]}

for i in range(2):
    for j in range(4):
        if 2*j+i >= 7:
            ax[i,j].set_xticks([])
            ax[i,j].set_yticks([])
            for spine in ax[i,j].spines.values():
                spine.set_visible(False)
            ax[i,j].minorticks_off()
            continue

        ax[i,j].set_title(names[2*j+i])
        ax[i,j].set_xlabel(r"$\log_{10}[M_*/M_\odot]$", fontsize=16)
        theta_vars = get_theta_1p_vars(2*j+i, mstar_test, 5)

        mu_fid, var_fid = model_smf(torch.asarray(theta_vars[2], dtype=torch.float)).mean.detach().numpy(), model_smf(torch.asarray(theta_vars[2], dtype=torch.float)).variance.detach().numpy()

        for n in range(5):
            mu_pred, var = model_smf(torch.asarray(theta_vars[n], dtype=torch.float)).mean.detach().numpy(), model_smf(torch.asarray(theta_vars[n], dtype=torch.float)).variance.detach().numpy()
            if n == 2:
                ax[i,j].plot(mstar_test, mu_pred - mu_fid, label="Fiducial", ls='--', color='k', lw=3)
            else:
                ax[i,j].plot(mstar_test, mu_pred - mu_fid, color=color_map[n], label=labels[n], lw=3)

for i in range(3):
    ax[0,i+1].set_yticklabels([])
    ax[1,i+1].set_yticklabels([])

# Y-axis labels on the leftmost column
ax[0,0].set_ylabel(r"$\Delta \log_{10}\Phi$", fontsize=16)
ax[1,0].set_ylabel(r"$\Delta \log_{10}\Phi$", fontsize=16)

# Legend in the empty panel
handles, leg_labels = ax[0,0].get_legend_handles_labels()
order = [0, 1, 2, 3, 4]
handles = [handles[k] for k in order]
leg_labels = [leg_labels[k] for k in order]

ax[1,3].legend(handles, leg_labels, loc='center', fontsize=14, frameon=False, title='Parameter Variation', title_fontsize=15)

plt.subplots_adjust(wspace=0.0, hspace=0.5)
plt.savefig("/cosmos_storage/home/fgmaion/MTNG-resims/Notebooks/plots/SMF_parameter_variation.pdf", bbox_inches='tight')